# Silver Registry

Single source of truth for the Silver runner (`99_Silver_Run_All`) and the cross-table validation
(`90_Silver_Cross_Table_Validation`). Plain Python only: no Spark, so it can be tested locally.

`readiness`
- `READY`: transformation rules are backed by the profiling recorded in the notebook.
- `NEEDS_REVIEW`: runnable, but the notebook lists rules/flags to confirm (see `docs/gold_governance_checks.md`, Source readiness).
- `NEEDS_PROFILING`: no reviewed implementation; profiling must be run and approved first. Not run by default.

Readiness is tracked on four separate axes. Passing local tests proves none of them.
- `code_prepared`: a notebook exists and is parameterised (always checked by tests).
- `semantics_reviewed`: field semantics, keys and rules were checked against the official MIMIC-IV
  documentation, as recorded in each notebook's profiling section. Values: `yes`, `partial`, `no`.
- `profiled`: profiling results are recorded in the notebook (counts/findings). `yes`, `partial`, `no`.
- `approved_to_publish`: a named reviewer approved publishing this table into the target schema. It
  is `False` for every table until someone deliberately changes it in a reviewed commit, together
  with `approved_by` and `approved_on`. The runner refuses a real run for any table without approval.

`readiness` is a coarse label only (`NEEDS_PROFILING` makes a table non-runnable by the runner); the four
axes above are authoritative.

`depends_on` lists Silver tables the notebook reads (in addition to its Bronze source).
`keys` is the grain key where profiling established uniqueness (`unique_key=True`), otherwise the
identifying columns whose NULLs are checked.

In [0]:
EXCLUDED = {}

DEFERRED = {}

# Ordered by domain; dependency order is computed, not assumed from this order.
REGISTRY = {
    # 01_Identity_Encounter
    "patients": dict(notebook="./01_Identity_Encounter/01_Patients", depends_on=[], readiness="READY",
                keys=['subject_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="no", approved_to_publish=False, flags=['age_deidentified_90_plus']),
    "admissions": dict(notebook="./01_Identity_Encounter/02_admissions", depends_on=[], readiness="READY",
                  keys=['hadm_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="partial", approved_to_publish=False, flags=['dq_invalid_admission_time', 'dq_invalid_ed_time', 'dq_death_before_admission', 'dq_death_after_discharge', 'dq_expire_deathtime_mismatch']),
    "icustays": dict(notebook="./01_Identity_Encounter/05_icustays", depends_on=['patients', 'admissions'], readiness="READY",
                keys=['stay_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="yes", approved_to_publish=False, flags=[]),
    "transfers": dict(notebook="./01_Identity_Encounter/03_transfers", depends_on=['patients', 'admissions', 'icustays'], readiness="READY",
                 keys=['transfer_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="yes", approved_to_publish=False, flags=['dq_invalid_transfer_time', 'dq_missing_outtime', 'dq_hadm_subject_mismatch']),
    "services": dict(notebook="./01_Identity_Encounter/04_services", depends_on=['patients', 'admissions'], readiness="NEEDS_PROFILING",
                keys=['hadm_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    # 02_Diagnoses_Procedures
    "d_icd_diagnoses": dict(notebook="./02_Diagnoses_Procedures/01_d_icd_diagnoses", depends_on=[], readiness="READY",
                       keys=['icd_code', 'icd_version'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "diagnoses_icd": dict(notebook="./02_Diagnoses_Procedures/02_diagnoses_icd", depends_on=[], readiness="READY",
                     keys=['subject_id', 'hadm_id', 'seq_num'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="no", approved_to_publish=False, flags=[]),
    "d_icd_procedures": dict(notebook="./02_Diagnoses_Procedures/03_d_icd_procedures", depends_on=[], readiness="READY",
                        keys=['icd_code', 'icd_version'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "procedures_icd": dict(notebook="./02_Diagnoses_Procedures/04_procedures_icd", depends_on=[], readiness="READY",
                      keys=['subject_id', 'hadm_id', 'seq_num'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="no", approved_to_publish=False, flags=[]),
    "drgcodes": dict(notebook="./02_Diagnoses_Procedures/05_drg_codes", depends_on=[], readiness="READY",
                keys=['subject_id', 'hadm_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="no", approved_to_publish=False, flags=[]),
    "d_hcpcs": dict(notebook="./02_Diagnoses_Procedures/06_d_hcpcs", depends_on=[], readiness="READY",
               keys=['code'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "hcpcsevents": dict(notebook="./02_Diagnoses_Procedures/07_hcpcsevents", depends_on=['d_hcpcs'], readiness="READY",
                   keys=['subject_id', 'hadm_id', 'seq_num'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "procedureevents": dict(notebook="./02_Diagnoses_Procedures/08_procedureevents", depends_on=[], readiness="READY",
                       keys=['stay_id', 'itemid'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="no", approved_to_publish=False, flags=['dq_invalid_procedure_time']),
    # 03_Labs_Observations (chartevents is deferred and not registered)
    "d_labitems": dict(notebook="./03_Labs_Observations/01_d_labitems", depends_on=[], readiness="READY",
                  keys=['itemid'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="no", approved_to_publish=False, flags=[]),
    "labevents": dict(notebook="./03_Labs_Observations/02_labevents", depends_on=[], readiness="READY",
                 keys=['labevent_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "d_items": dict(notebook="./03_Labs_Observations/03_d_items", depends_on=[], readiness="READY",
               keys=['itemid'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="yes", approved_to_publish=False, flags=[]),
    "datetimeevents": dict(notebook="./03_Labs_Observations/05_datetimeevents", depends_on=['icustays', 'd_items', 'caregiver'], readiness="READY",
                      keys=['stay_id', 'itemid'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="yes", approved_to_publish=False, flags=[]),
    "outputevents": dict(notebook="./03_Labs_Observations/06_outputevents", depends_on=['icustays', 'd_items', 'caregiver'], readiness="READY",
                    keys=['stay_id', 'itemid'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="yes", approved_to_publish=False, flags=['dq_negative_value', 'dq_storetime_before_charttime']),
    "microbiologyevents": dict(notebook="./03_Labs_Observations/07_microbiologyevents", depends_on=[], readiness="READY",
                          keys=['microevent_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "omr": dict(notebook="./03_Labs_Observations/08_omr", depends_on=[], readiness="READY",
           keys=['subject_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    # 04_Medication_Orders
    "prescriptions": dict(notebook="./04_Medication_Orders/01_prescriptions", depends_on=[], readiness="READY",
                     keys=['subject_id', 'hadm_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="partial", approved_to_publish=False, flags=[]),
    "pharmacy": dict(notebook="./04_Medication_Orders/02_pharmacy", depends_on=[], readiness="READY",
                keys=['subject_id', 'hadm_id'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="partial", approved_to_publish=False, flags=[]),
    "emar": dict(notebook="./04_Medication_Orders/03_emar", depends_on=[], readiness="READY",
            keys=['subject_id', 'emar_id', 'emar_seq'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="partial", approved_to_publish=False, flags=[]),
    "emar_detail": dict(notebook="./04_Medication_Orders/04_emar_details", depends_on=[], readiness="READY",
                   keys=['subject_id', 'emar_id', 'emar_seq'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="yes", approved_to_publish=False, flags=[]),
    "poe": dict(notebook="./04_Medication_Orders/05_poe", depends_on=['admissions'], readiness="READY",
            keys=['poe_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="yes", approved_to_publish=False, flags=['dq_hadm_not_in_admissions', 'dq_discontinue_of_poe_id_not_in_poe', 'dq_discontinued_by_poe_id_not_in_poe', 'dq_ordertime_before_previous_seq']),
    "poe_detail": dict(notebook="./04_Medication_Orders/06_poe_details", depends_on=[], readiness="READY",
                   keys=['poe_id', 'field_name'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="yes", approved_to_publish=False, flags=[]),
    "inputevents": dict(notebook="./04_Medication_Orders/07_inputevents", depends_on=[], readiness="READY",
                   keys=['subject_id', 'hadm_id', 'stay_id', 'itemid', 'orderid'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="partial", approved_to_publish=False, flags=[]),
    "ingredientevents": dict(notebook="./04_Medication_Orders/08_ingredientevents", depends_on=[], readiness="READY",
                        keys=['subject_id', 'hadm_id', 'stay_id', 'itemid', 'orderid'], unique_key=False, row_preserving=True, code_prepared=True, semantics_reviewed="partial", profiled="partial", approved_to_publish=False, flags=[]),
    # 05_People_Reference
    "provider": dict(notebook="./05_People_Reference/01_provider", depends_on=[], readiness="READY",
                keys=['provider_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="no", approved_to_publish=False, flags=[]),
    "caregiver": dict(notebook="./05_People_Reference/02_caregiver", depends_on=[], readiness="READY",
                 keys=['caregiver_id'], unique_key=True, row_preserving=True, code_prepared=True, semantics_reviewed="yes", profiled="yes", approved_to_publish=False, flags=[]),
}

In [0]:
def dependency_order(selected, registry=None):
    """Stable topological order of `selected` (dependencies outside the selection are ignored here)."""
    registry = REGISTRY if registry is None else registry
    ordered, done, visiting = [], set(), set()

    def visit(table):
        if table in done:
            return
        if table in visiting:
            raise ValueError(f"Dependency cycle involving {table}")
        visiting.add(table)
        for dependency in registry[table]["depends_on"]:
            if dependency in selected:
                visit(dependency)
        visiting.discard(table)
        done.add(table)
        ordered.append(table)

    for table in registry:                      # registry order keeps the result stable
        if table in selected:
            visit(table)
    return ordered


def build_plan(requested="all", registry=None):
    """Pure planning for a PREVIEW: order, readiness axes and why tables are not runnable.

    `requested="all"` is accepted for previews only; `require_executable` rejects it for real runs.
    """
    registry = REGISTRY if registry is None else registry
    names = list(registry) if requested == "all" else list(requested)
    if len(names) != len(set(names)):
        raise ValueError("Duplicate table names requested")
    if not names:
        raise ValueError("Select at least one table")

    skipped, selected = {}, set()
    for table in names:
        if table in EXCLUDED:
            if requested != "all":
                raise ValueError(f"{table} is excluded: {EXCLUDED[table]}")
            skipped[table] = EXCLUDED[table]
        elif table in DEFERRED:
            if requested != "all":
                raise ValueError(f"{table} is deferred: {DEFERRED[table]}")
            skipped[table] = DEFERRED[table]
        elif table not in registry:
            raise ValueError(f"Not in the Silver registry: {table}")
        else:
            selected.add(table)

    if requested == "all":
        for table, reason in {**EXCLUDED, **DEFERRED}.items():
            skipped.setdefault(table, reason)

    ordered = dependency_order(selected, registry)
    steps = []
    for i, table in enumerate(ordered):
        spec = registry[table]
        steps.append(dict(position=i + 1, table=table, notebook=spec["notebook"],
                          depends_on=list(spec["depends_on"]), readiness=spec["readiness"],
                          code_prepared=spec["code_prepared"], semantics_reviewed=spec["semantics_reviewed"],
                          profiled=spec["profiled"], approved_to_publish=spec["approved_to_publish"]))
    return dict(steps=steps, selected=selected, skipped=skipped, requested=requested)


def require_executable(plan):
    """Gate for a real run: explicit table selection and a recorded approval for every selected table.

    Tables that still need profiling are never run by the runner: run their notebook by hand after the
    profiling has been reviewed (the notebook has its own `profiling_approved` gate).
    """
    if plan["requested"] == "all":
        raise ValueError("A real run needs an explicit table selection; 'all' is preview-only")
    blocked = {}
    for step in plan["steps"]:
        reasons = []
        if step["readiness"] == "NEEDS_PROFILING":
            reasons.append("needs profiling: run its notebook manually after the profiling review")
        if not step["approved_to_publish"]:
            reasons.append("approved_to_publish is False in 00_Silver_Registry")
        if reasons:
            blocked[step["table"]] = "; ".join(reasons)
    if blocked:
        raise ValueError(f"Not approved for publication: {blocked}")
    return True